# ANN & DL — Lab Assignment 2 (Solved)

Bachelor-level reference implementation.

In [ ]:
# ANN & DL - Lab Assignment 2 (Solved)
# Bachelor-level reference implementation

import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 42
ASSET_DIR = Path('/mnt/data/lab2_solution_assets')
ASSET_DIR.mkdir(exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ============================================================
# TASK 1 - ANN Training and Hyperparameter Investigation
# ============================================================

# 1) Dataset preparation: train/validation/test + train-only scaling
X, y = load_iris(return_X_y=True)
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=SEED
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=SEED
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # fit only on training data
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

Xtr = torch.tensor(X_train, dtype=torch.float32)
ytr = torch.tensor(y_train, dtype=torch.long)
Xv = torch.tensor(X_val, dtype=torch.float32)
yv = torch.tensor(y_val, dtype=torch.long)
Xt = torch.tensor(X_test, dtype=torch.float32)
yt = torch.tensor(y_test, dtype=torch.long)

class IrisMLP(nn.Module):
    def __init__(self, initialization='he', dropout=0.0):
        super().__init__()
        self.fc1 = nn.Linear(4, 16)
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(16, 3)

        if initialization == 'he':
            nn.init.kaiming_uniform_(self.fc1.weight, nonlinearity='relu')
        elif initialization == 'xavier':
            nn.init.xavier_uniform_(self.fc1.weight)
        else:
            raise ValueError("initialization must be 'he' or 'xavier'")

        nn.init.zeros_(self.fc1.bias)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.zeros_(self.fc2.bias)

    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.dropout(x)
        return self.fc2(x)


def evaluate(model, X_tensor, y_tensor):
    model.eval()
    with torch.no_grad():
        logits = model(X_tensor)
        loss = nn.CrossEntropyLoss()(logits, y_tensor).item()
        acc = (logits.argmax(dim=1) == y_tensor).float().mean().item()
    return loss, acc


def train_mlp(initialization='he', lr=0.01, lr_strategy='constant',
              batch_size=16, epochs=10, l2=0.0, dropout=0.0,
              early_stopping=False, patience=5, seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = IrisMLP(initialization=initialization, dropout=dropout)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=l2)
    scheduler = None
    if lr_strategy == 'exponential':
        scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.90)

    generator = torch.Generator().manual_seed(seed)
    loader = DataLoader(
        TensorDataset(Xtr, ytr), batch_size=batch_size,
        shuffle=True, generator=generator
    )

    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [], 'lr': []
    }

    best_val_loss = float('inf')
    best_state = None
    wait = 0
    start = time.perf_counter()

    for epoch in range(epochs):
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0
        history['lr'].append(optimizer.param_groups[0]['lr'])

        for xb, yb in loader:
            optimizer.zero_grad()
            logits = model(xb)
            loss = nn.CrossEntropyLoss()(logits, yb)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * len(yb)
            correct += (logits.argmax(dim=1) == yb).sum().item()
            total += len(yb)

        val_loss, val_acc = evaluate(model, Xv, yv)
        history['train_loss'].append(running_loss / total)
        history['train_acc'].append(correct / total)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)

        if val_loss < best_val_loss - 1e-4:
            best_val_loss = val_loss
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if scheduler is not None:
            scheduler.step()

        if early_stopping and wait >= patience:
            break

    elapsed = time.perf_counter() - start

    if early_stopping and best_state is not None:
        model.load_state_dict(best_state)  # restore best validation weights

    final_val_loss, final_val_acc = evaluate(model, Xv, yv)
    test_loss, test_acc = evaluate(model, Xt, yt)
    result = {
        'epochs_run': len(history['val_loss']),
        'final_val_loss': final_val_loss,
        'final_val_acc': final_val_acc,
        'test_loss': test_loss,
        'test_acc': test_acc,
        'time_sec': elapsed,
    }
    return model, history, result


def run_task1_experiments():
    experiments = [
        ('Xavier init', dict(initialization='xavier', epochs=10)),
        ('He init', dict(initialization='he', epochs=10)),
        ('Constant LR', dict(initialization='he', lr_strategy='constant', epochs=10)),
        ('Exponential LR', dict(initialization='he', lr_strategy='exponential', epochs=10)),
        ('Batch size 8', dict(initialization='he', batch_size=8, epochs=10)),
        ('Batch size 16', dict(initialization='he', batch_size=16, epochs=10)),
        ('5 epochs', dict(initialization='he', epochs=5)),
        ('10 epochs', dict(initialization='he', epochs=10)),
        ('Baseline', dict(initialization='he', epochs=20)),
        ('L2', dict(initialization='he', epochs=20, l2=0.001)),
        ('Dropout', dict(initialization='he', epochs=20, dropout=0.20)),
        ('L2 + Dropout + Early stopping',
         dict(initialization='he', epochs=50, l2=0.001,
              dropout=0.20, early_stopping=True)),
    ]

    rows = []
    histories = {}
    models = {}
    for name, params in experiments:
        model, history, result = train_mlp(**params)
        rows.append({'Experiment': name, **result})
        histories[name] = history
        models[name] = model

    df = pd.DataFrame(rows)
    return df, histories, models


task1_results, task1_histories, task1_models = run_task1_experiments()

# Learning rate plot
lr_hist = task1_histories['Exponential LR']
plt.figure(figsize=(6.5, 4))
plt.plot(range(1, len(lr_hist['lr']) + 1), lr_hist['lr'], marker='o')
plt.xlabel('Epoch')
plt.ylabel('Learning rate')
plt.title('Exponential Learning-Rate Schedule')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'task1_lr_schedule.png', dpi=180)
plt.close()

# Learning curves for regularized + early-stopping model
curve_hist = task1_histories['L2 + Dropout + Early stopping']
epochs_axis = range(1, len(curve_hist['train_loss']) + 1)
plt.figure(figsize=(6.5, 4))
plt.plot(epochs_axis, curve_hist['train_loss'], label='Training loss')
plt.plot(epochs_axis, curve_hist['val_loss'], label='Validation loss')
plt.xlabel('Epoch')
plt.ylabel('Cross-entropy loss')
plt.title('Training and Validation Loss')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'task1_loss_curves.png', dpi=180)
plt.close()

plt.figure(figsize=(6.5, 4))
plt.plot(epochs_axis, curve_hist['train_acc'], label='Training accuracy')
plt.plot(epochs_axis, curve_hist['val_acc'], label='Validation accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'task1_accuracy_curves.png', dpi=180)
plt.close()

# ============================================================
# TASK 2 - Adaline and Madaline Networks
# ============================================================

class Adaline:
    """Batch Adaline using the delta/LMS rule and MSE loss."""
    def __init__(self, n_features, lr=0.2, epochs=30, seed=1):
        rng = np.random.default_rng(seed)
        self.w = rng.normal(0.0, 0.1, size=n_features)
        self.b = 0.0
        self.lr = lr
        self.epochs = epochs
        self.mse_history = []

    def net_input(self, X):
        return X @ self.w + self.b

    def predict(self, X):
        return np.where(self.net_input(X) >= 0, 1, -1)

    def fit(self, X, y):
        n = len(X)
        for _ in range(self.epochs):
            output = self.net_input(X)  # linear output during training
            error = y - output
            self.w += self.lr * (X.T @ error) / n
            self.b += self.lr * error.mean()
            self.mse_history.append(np.mean(error ** 2))
        return self

# Part A: AND bipolar truth table
X_and = np.array([[-1, -1], [-1, +1], [+1, -1], [+1, +1]], dtype=float)
y_and = np.array([-1, -1, -1, +1], dtype=float)

adaline_and = Adaline(n_features=2, lr=0.2, epochs=30, seed=1).fit(X_and, y_and)
and_predictions = adaline_and.predict(X_and)

plt.figure(figsize=(6.5, 4))
plt.plot(range(1, len(adaline_and.mse_history) + 1), adaline_and.mse_history, marker='o', markersize=3)
plt.xlabel('Epoch')
plt.ylabel('Mean squared error')
plt.title('Adaline Training on Bipolar AND')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'task2_adaline_mse.png', dpi=180)
plt.close()

# Part B: simple Madaline-style XOR network.
# Two hidden Adalines learn two positive XOR regions, then one Adaline combines them.
X_xor = np.array([[-1, -1], [-1, +1], [+1, -1], [+1, +1]], dtype=float)
y_xor = np.array([-1, +1, +1, -1], dtype=float)

# Desired hidden codes:
# H1 fires for (+1,-1); H2 fires for (-1,+1).
hidden_targets = np.array([
    [-1, -1],
    [-1, +1],
    [+1, -1],
    [-1, -1],
], dtype=float)

hidden_units = []
hidden_predictions = []
for j in range(2):
    unit = Adaline(n_features=2, lr=0.2, epochs=50, seed=2 + j)
    unit.fit(X_xor, hidden_targets[:, j])
    hidden_units.append(unit)
    hidden_predictions.append(unit.predict(X_xor))

H = np.column_stack(hidden_predictions)
output_unit = Adaline(n_features=2, lr=0.2, epochs=50, seed=9).fit(H, y_xor)
madaline_predictions = output_unit.predict(H)

# ============================================================
# TASK 3 - Pattern Association and Associative Memory
# ============================================================


def bipolar_sign(v):
    return np.where(v >= 0, 1, -1)

# Heteroassociative memory with orthogonal 4-bit inputs and 3-bit outputs
hetero_X = np.array([
    [1, 1, 1, 1],
    [1, 1, -1, -1],
    [1, -1, 1, -1],
], dtype=float)
hetero_Y = np.array([
    [1, -1, 1],
    [-1, 1, 1],
    [1, 1, -1],
], dtype=float)
hetero_W = sum(np.outer(y_vec, x_vec) for x_vec, y_vec in zip(hetero_X, hetero_Y))
hetero_recalled = np.array([bipolar_sign(hetero_W @ x) for x in hetero_X])

# Autoassociative memory using the pseudoinverse/projection rule
# Rows are stored bipolar patterns.
auto_patterns = np.array([
    [1, 1, 1, 1, -1, -1, -1, -1],
    [1, 1, -1, -1, 1, 1, -1, -1],
    [1, -1, 1, -1, 1, -1, 1, -1],
], dtype=float)
auto_W = auto_patterns.T @ np.linalg.pinv(auto_patterns @ auto_patterns.T) @ auto_patterns


def auto_recall(query):
    return bipolar_sign(auto_W @ query)


def flip_bits(pattern, indices):
    q = pattern.copy()
    q[list(indices)] *= -1
    return q

# Cases include both successful and failed noisy recalls.
auto_cases = [
    (0, (), 'clean'),
    (0, (0,), 'flip bit 1'),
    (0, (0, 1), 'flip bits 1-2'),
    (1, (), 'clean'),
    (1, (0,), 'flip bit 1'),
    (1, (0, 1), 'flip bits 1-2'),
    (2, (), 'clean'),
    (2, (0, 1), 'flip bits 1-2'),
    (2, (0, 1, 2), 'flip bits 1-3'),
]

auto_rows = []
for pattern_idx, indices, noise_name in auto_cases:
    stored = auto_patterns[pattern_idx].astype(int)
    query = flip_bits(auto_patterns[pattern_idx], indices).astype(int)
    recalled = auto_recall(query).astype(int)
    auto_rows.append({
        'Pattern': f'P{pattern_idx + 1}',
        'Stored pattern': stored.tolist(),
        'Query / noise': noise_name,
        'Recalled output': recalled.tolist(),
        'Exact match': bool(np.array_equal(recalled, stored)),
    })
auto_results = pd.DataFrame(auto_rows)

# ============================================================
# TASK 4 - Hopfield Network
# ============================================================

# Construct 3 orthogonal length-16 bipolar patterns from a Hadamard matrix.
H16 = np.array([[1.0]])
for _ in range(4):
    H16 = np.block([[H16, H16], [H16, -H16]])
hopfield_patterns = H16[[1, 2, 4]].astype(int)
N = hopfield_patterns.shape[1]

hopfield_W = sum(np.outer(p, p) for p in hopfield_patterns) / N
np.fill_diagonal(hopfield_W, 0.0)


def hopfield_energy(state):
    state = np.asarray(state)
    return float(-0.5 * state @ hopfield_W @ state)


def hopfield_recall(query, max_sweeps=20, record_changed_only=False):
    """Asynchronous cyclic updates. If weighted input is 0, keep old state."""
    state = np.array(query, dtype=int).copy()
    energies = [hopfield_energy(state)]

    for sweep in range(max_sweeps):
        changed_in_sweep = False
        for i in range(N):
            h_i = hopfield_W[i] @ state
            if h_i > 0:
                new_value = 1
            elif h_i < 0:
                new_value = -1
            else:
                new_value = state[i]  # tie: keep current value

            changed = new_value != state[i]
            if changed:
                state[i] = new_value
                changed_in_sweep = True
                energies.append(hopfield_energy(state))
            elif not record_changed_only:
                energies.append(hopfield_energy(state))

        if not changed_in_sweep:
            break

    return state, energies, sweep + 1

# Recall success rates. For reproducibility, the first k bits are flipped.
hopfield_rows = []
for flips in [0, 1, 3, 6, 9, 10]:
    successes = 0
    details = []
    for p in hopfield_patterns:
        q = p.copy()
        q[:flips] *= -1
        recalled, _, _ = hopfield_recall(q)
        ok = np.array_equal(recalled, p)
        successes += int(ok)
        details.append(ok)
    hopfield_rows.append({
        'Flipped bits': flips,
        'P1 success': details[0],
        'P2 success': details[1],
        'P3 success': details[2],
        'Success rate': successes / len(hopfield_patterns),
    })
hopfield_results = pd.DataFrame(hopfield_rows)

# Energy example: P1 with first 3 bits flipped.
energy_query = hopfield_patterns[0].copy()
energy_query[:3] *= -1
energy_recalled, energy_trace, energy_sweeps = hopfield_recall(
    energy_query, record_changed_only=True
)

plt.figure(figsize=(6.5, 4))
plt.plot(range(len(energy_trace)), energy_trace, marker='o')
plt.xlabel('State changes during asynchronous recall')
plt.ylabel('Hopfield energy')
plt.title('Hopfield Energy During Recall')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(ASSET_DIR / 'task4_energy.png', dpi=180)
plt.close()

# ============================================================
# Console summary
# ============================================================

if __name__ == '__main__':
    print('TASK 1: Dataset sizes')
    print('Train:', len(X_train), 'Validation:', len(X_val), 'Test:', len(X_test))
    print('\nTASK 1: Experiment summary')
    print(task1_results.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

    print('\nTASK 2A: Adaline AND')
    print('Weights:', np.round(adaline_and.w, 4))
    print('Bias:', round(adaline_and.b, 4))
    print('Predictions:', and_predictions.astype(int))
    print('Final MSE:', round(adaline_and.mse_history[-1], 6))

    print('\nTASK 2B: Madaline-style XOR')
    for i, unit in enumerate(hidden_units, start=1):
        print(f'Hidden H{i} weights:', np.round(unit.w, 4), 'bias:', round(unit.b, 4))
    print('Hidden outputs:\n', H.astype(int))
    print('Output weights:', np.round(output_unit.w, 4), 'bias:', round(output_unit.b, 4))
    print('Predictions:', madaline_predictions.astype(int))

    print('\nTASK 3A: Heteroassociative weight matrix')
    print(hetero_W.astype(int))
    print('Recalled outputs:\n', hetero_recalled.astype(int))

    print('\nTASK 3B: Autoassociative results')
    print(auto_results.to_string(index=False))

    print('\nTASK 4: Stored patterns')
    print(hopfield_patterns)
    print('\nTASK 4: 16 * Hopfield weight matrix (so W = shown matrix / 16)')
    print(np.rint(hopfield_W * 16).astype(int))
    print('\nTASK 4: Recall results')
    print(hopfield_results.to_string(index=False))
    print('\nTASK 4: Energy trace')
    print([round(e, 4) for e in energy_trace])
